In [1]:
!pip install python-doctr -q
!pip install torch torchvision -q
!pip install opencv-python-headless matplotlib -q
!pip install python-docx -q
!pip install jiwer -q
!pip install huggingface_hub -q
!pip install google-cloud-bigquery-storage -q
!pip install -U -q "google-genai"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 33.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.9/288.9 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 89.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 64.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.4/304.4 kB 11.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 750.9/750.9 kB 12.3 MB/s eta 0:00:0000:01


In [2]:
import os
import re
import numpy as np
from PIL import Image
from doctr.models import ocr_predictor
from doctr.io import DocumentFile
from google import genai
from jiwer import cer, wer
import pandas as pd
import unicodedata

In [3]:
# Paths
input_folder  = "/kaggle/input/datasets/satarupadeb/renaissance/preprocessed-1628"
output_folder = "/kaggle/working/ocr_results"
os.makedirs(output_folder, exist_ok=True)

# List images
image_files = sorted([
    f for f in os.listdir(input_folder)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])
print(f"Found {len(image_files)} images:")
for f in image_files:
    print(f"  {f}")

Found 12 images:
  img_0001.jpg
  img_0002.jpg
  img_0003.jpg
  img_0004.jpg
  img_0005.jpg
  img_0006.jpg
  img_0007.jpg
  img_0008.jpg
  img_0009.jpg
  img_0010.jpg
  img_0011.jpg
  img_0012.jpg


In [4]:
# Load docTR 
print("Loading docTR model...")
model = ocr_predictor(
    det_arch='db_resnet50',
    reco_arch='parseq',
    pretrained=True,
    assume_straight_pages=True
)
print("Model loaded.")

Loading docTR model...


  0%|          | 0/102021912 [00:00<?, ?it/s]

  0%|          | 0/95457349 [00:00<?, ?it/s]

Model loaded.


In [5]:
# Marginalia filter 
def filter_marginalia(text):
    """
    Remove short scripture references .
    """
    lines = text.split('\n')
    filtered = []
    for line in lines:
        stripped = line.strip()
        if len(stripped) < 4:
            continue
        if re.match(r'^[A-Za-z]{1,6}\.?\s*\d', stripped) and len(stripped) < 20:
            continue
        filtered.append(line)
    return '\n'.join(filtered)

# Run OCR on all images 
ocr_results = {}

for img_name in image_files:
    img_path = os.path.join(input_folder, img_name)
    print(f"Processing {img_name}...")

    doc      = DocumentFile.from_images([img_path])
    result   = model(doc)
    raw_text = result.render()
    clean    = filter_marginalia(raw_text)

    ocr_results[img_name] = clean
    print(f"  → {len(clean)} characters")

print(f"\nOCR complete. {len(ocr_results)} images processed.")

Processing img_0001.jpg...
  → 1066 characters
Processing img_0002.jpg...
  → 2985 characters
Processing img_0003.jpg...
  → 3260 characters
Processing img_0004.jpg...
  → 3535 characters
Processing img_0005.jpg...
  → 3312 characters
Processing img_0006.jpg...
  → 2818 characters
Processing img_0007.jpg...
  → 3223 characters
Processing img_0008.jpg...
  → 3233 characters
Processing img_0009.jpg...
  → 3237 characters
Processing img_0010.jpg...
  → 3269 characters
Processing img_0011.jpg...
  → 3145 characters
Processing img_0012.jpg...
  → 1698 characters

OCR complete. 12 images processed.


In [6]:
#  Save raw OCR output 
raw_path = os.path.join(output_folder, "raw_ocr_output.txt")

with open(raw_path, 'w', encoding='utf-8') as f:
    for img_name in sorted(ocr_results.keys()):
        f.write(f"\n{'='*50}\n")
        f.write(f"IMAGE: {img_name}\n")
        f.write(f"{'='*50}\n")
        f.write(ocr_results[img_name])
        f.write("\n")

print(f"Saved to {raw_path}")

# Preview result 
first = sorted(ocr_results.keys())[0]
print(f"\n--- Preview: {first} ---")
print(ocr_results[first][:600])

Saved to /kaggle/working/ocr_results/raw_ocr_output.txt

--- Preview: img_0001.jpg ---
P.do.s
eftanod L olde
obsoyedpha
Molicibonaos SUP
E tynab oba
upasus,babinez
23 Igsh redoonobel
isoy Yssoldon sb
Y,enussoprention
copelinos siod
-ocelobasuoll
@ 26130 Ye20bezoz
plobeb sbsnorg
moiss.oborordol
biusdsb 2513321
el obiran ul Yer
ponnobogallob
0701031 303 allsi IDE
cinodesb a
edoib
avion ozib
nob a
orloib, la
-zelso siroup sup,
Rail Is prolliv
orlaib offoussbe obisangasic pus
up Kestid dnoash
DOT NACATALINAD E
210 eny Zarate,viuda obri a P a a
Laçarragay
deMarco
AntoniAntode Caycodo.yelfotior Y
20111 : 2odoib eal Fifcal. S - 1100000 El coloil
FR isflamaloanomale a : :
sbub obaup on:


In [7]:
# Rule-based correction (baseline)

def rule_based_correct(text):
    """
    Fix systematic errors from 18th century Spanish printing.
    Based on rules in the transcription notes.
  
    """
    replacements = [
        (r'\bef([aeiou])', r'es\1'),   # efta → esta
        (r'\baf([aeiou])', r'as\1'),   # afsi → assi
        (r'\bfo([aeiou])', r'so\1'),   # folo → solo
        (r'\bfi([aeiou])', r'si\1'),   # fino → sino
        (r'\bfe([aeiou])', r'se\1'),   # fea  → sea
        (r'\bfu([aeiou])', r'su\1'),   # fus  → sus
        (r'ç', 'z'),                    # ç → z
        (r'\bvna\b', 'una'),
        (r'\bvn\b', 'un'),
    ]
    for pattern, replacement in replacements:
        text = re.sub(pattern, replacement, text)
    return text

# Apply to all OCR results
rule_based_results = {}
for img_name, raw_text in ocr_results.items():
    rule_based_results[img_name] = rule_based_correct(raw_text)
    print(f"Rule-based corrected: {img_name}")

print("Done.")

Rule-based corrected: img_0001.jpg
Rule-based corrected: img_0002.jpg
Rule-based corrected: img_0003.jpg
Rule-based corrected: img_0004.jpg
Rule-based corrected: img_0005.jpg
Rule-based corrected: img_0006.jpg
Rule-based corrected: img_0007.jpg
Rule-based corrected: img_0008.jpg
Rule-based corrected: img_0009.jpg
Rule-based corrected: img_0010.jpg
Rule-based corrected: img_0011.jpg
Rule-based corrected: img_0012.jpg
Done.


In [8]:
from docx import Document
import re

DOCX_PATH = "/kaggle/input/datasets/satarupadeb/renaissance/transcripts/PORCONES.23.5 - 1628 transcription.docx"

def extract_transcript(docx_path):
    doc = Document(docx_path)
    text = "\n".join([p.text for p in doc.paragraphs])

    splits = re.split(r'PDF p(\d+)', text, flags=re.IGNORECASE)

    transcript = {}

    for i in range(1, len(splits), 2):
        page_num = int(splits[i])
        content = splits[i+1].strip()

        if page_num not in transcript:
            transcript[page_num] = []

        transcript[page_num].append(content)

    return transcript

transcript = extract_transcript(DOCX_PATH)

# debug
for k, v in transcript.items():
    print(f"Page {k}: {len(v)} segments")

Page 1: 1 segments
Page 2: 1 segments
Page 3: 3 segments
Page 4: 1 segments


In [9]:
page_to_image = {}

for page_num in transcript.keys():
    img_name = f"img_{page_num:04d}.jpg"
    
    if img_name in image_files:   
        page_to_image[page_num] = img_name

print("Mapping:")
for k, v in page_to_image.items():
    print(f"PDF p{k} → {v}")

Mapping:
PDF p1 → img_0001.jpg
PDF p2 → img_0002.jpg
PDF p3 → img_0003.jpg
PDF p4 → img_0004.jpg


In [10]:
print("Transcript pages:", sorted(transcript.keys()))
print("Image files:", image_files[:5])

Transcript pages: [1, 2, 3, 4]
Image files: ['img_0001.jpg', 'img_0002.jpg', 'img_0003.jpg', 'img_0004.jpg', 'img_0005.jpg']


In [11]:
!pip install jiwer -q

In [12]:
from jiwer import cer, wer
import pandas as pd
import re


In [13]:
def normalize_text(text):
    # join hyphenated line breaks
    text = re.sub(r'-\n', '', text)

    # lowercase
    text = text.lower().strip()

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    # remove accents (except ñ)
    normalized = []
    for c in unicodedata.normalize('NFD', text):
        if unicodedata.category(c) == 'Mn' and c != '\u0303':
            continue
        normalized.append(c)

    return unicodedata.normalize('NFC', ''.join(normalized))

In [14]:
# Evaluation: Raw OCR vs Rule-based
from jiwer import cer, wer
import pandas as pd

results_baseline = []

for page_num, img_name in page_to_image.items():
    transcript_text = "\n".join(transcript[page_num])

    gt       = normalize_text(transcript_text)
    raw      = normalize_text(ocr_results[img_name])
    ruled    = normalize_text(rule_based_results[img_name])

    results_baseline.append({
        "Page":            page_num,
        "Image":           img_name,
        "CER (Raw OCR)":   round(cer(gt, raw),   4),
        "WER (Raw OCR)":   round(wer(gt, raw),   4),
        "CER (Rule-based)": round(cer(gt, ruled), 4),
        "WER (Rule-based)": round(wer(gt, ruled), 4),
    })

df_baseline = pd.DataFrame(results_baseline).sort_values("Page")

# Average row
avg = {
    "Page":             "AVG",
    "Image":            "-",
    "CER (Raw OCR)":    round(df_baseline["CER (Raw OCR)"].mean(),    4),
    "WER (Raw OCR)":    round(df_baseline["WER (Raw OCR)"].mean(),    4),
    "CER (Rule-based)": round(df_baseline["CER (Rule-based)"].mean(), 4),
    "WER (Rule-based)": round(df_baseline["WER (Rule-based)"].mean(), 4),
}
df_baseline = pd.concat(
    [df_baseline, pd.DataFrame([avg])], ignore_index=True
)

print("=" * 65)
print("BASELINE EVALUATION (Raw OCR vs Rule-based Correction)")
print("=" * 65)
print(df_baseline.to_string(index=False))

BASELINE EVALUATION (Raw OCR vs Rule-based Correction)
Page        Image  CER (Raw OCR)  WER (Raw OCR)  CER (Rule-based)  WER (Rule-based)
   1 img_0001.jpg         1.4208         1.6889            1.4167            1.6889
   2 img_0002.jpg         1.1945         0.9631            1.1945            0.9631
   3 img_0003.jpg         0.7506         0.9721            0.7504            0.9712
   4 img_0004.jpg         1.2573         1.0507            1.2578            1.0507
 AVG            -         1.1558         1.1687            1.1549            1.1685


In [15]:

GEMINI_API_KEY = ""
client = genai.Client(api_key=GEMINI_API_KEY)

# Prompt builder
def build_prompt(raw_text):
    return f"""You are an expert in 17th century Spanish printed texts.

Correct OCR errors from a historical Spanish book.

Apply these rules:
1. Long-s misread as 'f': afsi→assi, efta→esta, folo→solo, fino→sino
2. 'ç' → 'z'
3. 'u' and 'v' are interchangeable — choose correct form based on context
4. Fix spacing issues and OCR noise
5. Do NOT modernize spelling (keep forms like 'assi', 'Reyno')
6. Do NOT add or remove content

Return ONLY corrected text.

RAW OCR:
{raw_text}"""

# correction function
def gemini_correct(raw_text):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=build_prompt(raw_text),
        config={
            "temperature": 0.1
        }
    )
    return response.text

# Run correction on all OCR outputs
print("Running Gemini correction...")
gemini_results = {}

for img_name, raw_text in ocr_results.items():
    print(f"Correcting {img_name}...")
    
    try:
        gemini_results[img_name] = gemini_correct(raw_text)
    except Exception as e:
        print(f"Error on {img_name}: {e}")
        gemini_results[img_name] = raw_text  # fallback

print("Gemini correction done ")

# result
first = sorted(gemini_results.keys())[0]

print("\n--- RAW OCR ---\n")
print(ocr_results[first][:500])

print("\n--- GEMINI CORRECTED ---\n")
print(gemini_results[first][:500])

Running Gemini correction...
Correcting img_0001.jpg...
Correcting img_0002.jpg...
Correcting img_0003.jpg...
Correcting img_0004.jpg...
Correcting img_0005.jpg...
Correcting img_0006.jpg...
Correcting img_0007.jpg...
Correcting img_0008.jpg...
Correcting img_0009.jpg...
Error on img_0009.jpg: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 29.981404835s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'typ

In [16]:
# Evaluate Gemini results
results_gemini = []

for page_num, img_name in page_to_image.items():
    transcript_text = "\n".join(transcript[page_num])

    gt     = normalize_text(transcript_text)
    raw    = normalize_text(ocr_results[img_name])
    gem    = normalize_text(gemini_results[img_name])

    results_gemini.append({
        "Page":           page_num,
        "Image":          img_name,
        "CER (Raw OCR)":  round(cer(gt, raw), 4),
        "WER (Raw OCR)":  round(wer(gt, raw), 4),
        "CER (Gemini)":   round(cer(gt, gem), 4),
        "WER (Gemini)":   round(wer(gt, gem), 4),
    })

df_gemini = pd.DataFrame(results_gemini).sort_values("Page")

avg_g = {
    "Page":          "AVG",
    "Image":         "-",
    "CER (Raw OCR)": round(df_gemini["CER (Raw OCR)"].mean(), 4),
    "WER (Raw OCR)": round(df_gemini["WER (Raw OCR)"].mean(), 4),
    "CER (Gemini)":  round(df_gemini["CER (Gemini)"].mean(),  4),
    "WER (Gemini)":  round(df_gemini["WER (Gemini)"].mean(),  4),
}
df_gemini = pd.concat(
    [df_gemini, pd.DataFrame([avg_g])], ignore_index=True
)

print("=" * 60)
print("GEMINI EVALUATION")
print("=" * 60)
print(df_gemini.to_string(index=False))

GEMINI EVALUATION
Page        Image  CER (Raw OCR)  WER (Raw OCR)  CER (Gemini)  WER (Gemini)
   1 img_0001.jpg         1.4208         1.6889        0.9250        1.2000
   2 img_0002.jpg         1.1945         0.9631        0.9354        1.1946
   3 img_0003.jpg         0.7506         0.9721        0.7067        0.9014
   4 img_0004.jpg         1.2573         1.0507        0.6185        0.8479
 AVG            -         1.1558         1.1687        0.7964        1.0360


In [17]:
def build_prompt(raw_text):
    return f"""You are an expert in 18th century Spanish printed texts and historical OCR correction.
You are correcting OCR output from a 1740 Spanish book printed in Gerona, Spain.

Apply these correction rules strictly:
1. The long-s (ſ) was misread as 'f' — fix where it makes Spanish sense:
   e.g. 'afsi' → 'assi', 'efta' → 'esta', 'folo' → 'solo', 'fino' → 'sino',
        'fea' → 'sea', 'fus' → 'sus', 'fu' → 'su', 'fon' → 'son'
2. 'ç' is always modern 'z' — e.g. 'haçer' → 'hazer'
3. 'u' and 'v' are interchangeable — use whichever makes Spanish sense:
   e.g. 'vna' → 'una', 'vn' → 'un', 'vuestra' stays 'vuestra'
4. Fix garbled characters and missing spaces between words
5. Fix broken words caused by line-end hyphenation if obvious
6. Do NOT modernize spelling — preserve period forms like:
   'assi', 'vuestra', 'Reyno', 'christiana', 'dixo', 'hazer', 'mesmo'
7. Do NOT add or remove any content — only fix OCR errors
8. Do NOT add explanations, notes, or commentary

Return ONLY the corrected text, nothing else.

RAW OCR TEXT:
{raw_text}"""

In [18]:
from huggingface_hub import InferenceClient

HF_API_KEY = ""   

def llama_correct(raw_text):
    client = InferenceClient(
        model="meta-llama/Meta-Llama-3-8B-Instruct",
        token=HF_API_KEY
    )
    response = client.chat_completion(
        messages=[
            {
                "role": "system",
                "content": "You are an expert in 18th century Spanish printed texts. Correct OCR errors only. Return only the corrected text."
            },
            {
                "role": "user",
                "content": build_prompt(raw_text)
            }
        ],
        max_tokens=2048,
        temperature=0.1
    )
    return response.choices[0].message.content

# Run Llama correction
print("Running Llama correction...")
llama_results = {}

for img_name, raw_text in ocr_results.items():
    print(f"  Correcting {img_name}...")
    try:
        llama_results[img_name] = llama_correct(raw_text)
    except Exception as e:
        print(f"  Error on {img_name}: {e}")
        llama_results[img_name] = raw_text   # fallback

print("Llama correction done.")

Running Llama correction...
  Correcting img_0001.jpg...
  Correcting img_0002.jpg...
  Correcting img_0003.jpg...
  Correcting img_0004.jpg...
  Correcting img_0005.jpg...
  Correcting img_0006.jpg...
  Correcting img_0007.jpg...
  Correcting img_0008.jpg...
  Correcting img_0009.jpg...
  Correcting img_0010.jpg...
  Correcting img_0011.jpg...
  Correcting img_0012.jpg...
Llama correction done.


In [19]:
results_llama = []

for page_num, img_name in page_to_image.items():
    transcript_text = "\n".join(transcript[page_num])

    gt    = normalize_text(transcript_text)
    raw   = normalize_text(ocr_results[img_name])
    llm   = normalize_text(llama_results[img_name])

    results_llama.append({
        "Page":          page_num,
        "Image":         img_name,
        "CER (Raw OCR)": round(cer(gt, raw), 4),
        "WER (Raw OCR)": round(wer(gt, raw), 4),
        "CER (Llama)":   round(cer(gt, llm), 4),
        "WER (Llama)":   round(wer(gt, llm), 4),
    })

df_llama = pd.DataFrame(results_llama).sort_values("Page")

avg_l = {
    "Page":          "AVG",
    "Image":         "-",
    "CER (Raw OCR)": round(df_llama["CER (Raw OCR)"].mean(), 4),
    "WER (Raw OCR)": round(df_llama["WER (Raw OCR)"].mean(), 4),
    "CER (Llama)":   round(df_llama["CER (Llama)"].mean(),   4),
    "WER (Llama)":   round(df_llama["WER (Llama)"].mean(),   4),
}
df_llama = pd.concat(
    [df_llama, pd.DataFrame([avg_l])], ignore_index=True
)

print("=" * 60)
print("LLAMA EVALUATION")
print("=" * 60)
print(df_llama.to_string(index=False))

LLAMA EVALUATION
Page        Image  CER (Raw OCR)  WER (Raw OCR)  CER (Llama)  WER (Llama)
   1 img_0001.jpg         1.4208         1.6889       1.4292       1.6889
   2 img_0002.jpg         1.1945         0.9631       1.2063       1.0168
   3 img_0003.jpg         0.7506         0.9721       0.7224       0.9321
   4 img_0004.jpg         1.2573         1.0507       1.2542       1.0535
 AVG            -         1.1558         1.1687       1.1530       1.1728
